In [1]:
import os
import joblib
import numpy as np
import pandas as pd

from scipy.sparse import load_npz
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix
)

In [1]:
import transformers
print(transformers.__version__)
from transformers import TrainingArguments
import inspect
print("evaluation_strategy" in inspect.signature(TrainingArguments.__init__).parameters)
print("eval_strategy" in inspect.signature(TrainingArguments.__init__).parameters)


4.57.6
False
True


In [2]:
os.makedirs("results/metrics", exist_ok=True)
os.makedirs("results/confusion_matrices", exist_ok=True)


In [3]:
X_liar_train = load_npz("results/features/X_liar_train.npz")
X_liar_valid = load_npz("results/features/X_liar_valid.npz")
X_liar_test  = load_npz("results/features/X_liar_test.npz")

y_liar_train = joblib.load("results/features/y_liar_train.pkl")
y_liar_valid = joblib.load("results/features/y_liar_valid.pkl")
y_liar_test  = joblib.load("results/features/y_liar_test.pkl")

tfidf_liar = joblib.load("results/features/tfidf_liar.joblib")

print("Loaded LIAR features:")
print("Train:", X_liar_train.shape, "Test:", X_liar_test.shape)


Loaded LIAR features:
Train: (10240, 5000) Test: (1267, 5000)


In [4]:
X_isot_train = load_npz("results/features/X_isot_train.npz")
X_isot_valid = load_npz("results/features/X_isot_valid.npz")
X_isot_test  = load_npz("results/features/X_isot_test.npz")

y_isot_train = joblib.load("results/features/y_isot_train.pkl")
y_isot_valid = joblib.load("results/features/y_isot_valid.pkl")
y_isot_test  = joblib.load("results/features/y_isot_test.pkl")

tfidf_isot = joblib.load("results/features/tfidf_isot.joblib")

# Load raw ISOT splits (needed for cross-domain transforms)
isot_train = pd.read_csv("results/features/isot_train.csv")
isot_valid = pd.read_csv("results/features/isot_valid.csv")
isot_test  = pd.read_csv("results/features/isot_test.csv")

print("Loaded ISOT features:")
print("Train:", X_isot_train.shape, "Test:", X_isot_test.shape)

Loaded ISOT features:
Train: (31428, 5000) Test: (6735, 5000)


In [5]:
import numpy as np

print("LIAR train unique:", np.unique(y_liar_train))
print("LIAR test  unique:", np.unique(y_liar_test))

print("ISOT train unique:", np.unique(y_isot_train))
print("ISOT test  unique:", np.unique(y_isot_test))


LIAR train unique: [0 1]
LIAR test  unique: [0 1]
ISOT train unique: [0 1]
ISOT test  unique: [0 1]


In [6]:
# Load LIAR test text safely
liar_test_df = pd.read_csv("../data/liar/liar_test_clean.csv")
liar_test_text = liar_test_df["statement"].fillna("").astype(str)

# Ensure ISOT test statements are safe (from results/features/isot_test.csv)
isot_test_text = isot_test["statement"].fillna("").astype(str)

# Cross-domain transformations
X_isot_test_as_liar = tfidf_liar.transform(isot_test_text)
X_liar_test_as_isot = tfidf_isot.transform(liar_test_text)

print("Cross-domain matrices:")
print("ISOT test as LIAR vocab:", X_isot_test_as_liar.shape)
print("LIAR test as ISOT vocab:", X_liar_test_as_isot.shape)

# Optional sanity checks
print("NaNs in ISOT test:", isot_test["statement"].isna().sum())
print("NaNs in LIAR test:", liar_test_df["statement"].isna().sum())


Cross-domain matrices:
ISOT test as LIAR vocab: (6735, 5000)
LIAR test as ISOT vocab: (1267, 5000)
NaNs in ISOT test: 104
NaNs in LIAR test: 0


In [7]:
# Evaluation helper
def evaluate_model(model, X, y, experiment_name):
    preds = model.predict(X)
    acc = accuracy_score(y, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(y, preds, average="binary")
    cm = confusion_matrix(y, preds)
    return {
        "Experiment": experiment_name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1": f1,
        "ConfusionMatrix": cm
    }

In [8]:
# Train on LIAR, evaluate LIAR->LIAR and LIAR->ISOT
results = []

# Logistic Regression (LIAR)
lr_liar = LogisticRegression(max_iter=2000)
lr_liar.fit(X_liar_train, y_liar_train)

# Linear SVM (LIAR)
svm_liar = LinearSVC(dual="auto")
svm_liar.fit(X_liar_train, y_liar_train)

# In-domain: LIAR -> LIAR
results.append(evaluate_model(lr_liar, X_liar_test, y_liar_test, "LR (LIAR→LIAR)"))
results.append(evaluate_model(svm_liar, X_liar_test, y_liar_test, "SVM (LIAR→LIAR)"))

# Cross-domain: LIAR -> ISOT (use LIAR vocabulary on ISOT)
results.append(evaluate_model(lr_liar, X_isot_test_as_liar, y_isot_test, "LR (LIAR→ISOT)"))
results.append(evaluate_model(svm_liar, X_isot_test_as_liar, y_isot_test, "SVM (LIAR→ISOT)"))

print("Completed LIAR-trained experiments.")


Completed LIAR-trained experiments.


In [9]:
# Train on ISOT, evaluate ISOT->ISOT and ISOT->LIAR
# Logistic Regression (ISOT)
lr_isot = LogisticRegression(max_iter=2000)
lr_isot.fit(X_isot_train, y_isot_train)

# Linear SVM (ISOT)
svm_isot = LinearSVC(dual="auto")
svm_isot.fit(X_isot_train, y_isot_train)

# In-domain: ISOT -> ISOT
results.append(evaluate_model(lr_isot, X_isot_test, y_isot_test, "LR (ISOT→ISOT)"))
results.append(evaluate_model(svm_isot, X_isot_test, y_isot_test, "SVM (ISOT→ISOT)"))

# Cross-domain: ISOT -> LIAR (use ISOT vocabulary on LIAR)
results.append(evaluate_model(lr_isot, X_liar_test_as_isot, y_liar_test, "LR (ISOT→LIAR)"))
results.append(evaluate_model(svm_isot, X_liar_test_as_isot, y_liar_test, "SVM (ISOT→LIAR)"))

print("Completed ISOT-trained experiments.")

Completed ISOT-trained experiments.


In [10]:
# Build metrics table
metrics_df = pd.DataFrame([{
    "Experiment": r["Experiment"],
    "Accuracy": r["Accuracy"],
    "Precision": r["Precision"],
    "Recall": r["Recall"],
    "F1": r["F1"]
} for r in results]).sort_values("Experiment").reset_index(drop=True)

metrics_df

,Experiment,Accuracy,Precision,Recall,F1
0,LR (ISOT→ISOT),0.989458,0.986977,0.990971,0.988970
1,LR (ISOT→LIAR),0.371744,0.666667,0.024814,0.047847
2,LR (LIAR→ISOT),0.479733,0.477724,0.974782,0.641204
3,LR (LIAR→LIAR),0.640884,0.655723,0.916873,0.764615
4,SVM (ISOT→ISOT),0.994803,0.995632,0.993462,0.994546
5,SVM (ISOT→LIAR),0.370166,0.642857,0.022333,0.043165
6,SVM (LIAR→ISOT),0.494135,0.483572,0.893524,0.627528
7,SVM (LIAR→LIAR),0.604578,0.664865,0.763027,0.710572


In [11]:
# Save metrics + confusion matrices
metrics_path = "results/metrics/ml_baselines_metrics.csv"
metrics_df.to_csv(metrics_path, index=False)
print("Saved metrics to:", metrics_path)

for r in results:
    cm = r["ConfusionMatrix"]
    exp_name = (
        r["Experiment"]
        .replace("→", "_to_")
        .replace(" ", "_")
        .replace("(", "")
        .replace(")", "")
    )
    cm_path = f"results/confusion_matrices/{exp_name}_cm.npy"
    np.save(cm_path, cm)

print("Saved confusion matrices to results/confusion_matrices/")

Saved metrics to: results/metrics/ml_baselines_metrics.csv
Saved confusion matrices to results/confusion_matrices/


In [12]:
# Show cross-domain F1 drops
def get_metric(exp, metric="F1"):
    return metrics_df.loc[metrics_df["Experiment"] == exp, metric].values[0]

print("F1 drop LR (LIAR→LIAR vs LIAR→ISOT):",
      round(get_metric("LR (LIAR→LIAR)") - get_metric("LR (LIAR→ISOT)"), 4))

print("F1 drop LR (ISOT→ISOT vs ISOT→LIAR):",
      round(get_metric("LR (ISOT→ISOT)") - get_metric("LR (ISOT→LIAR)"), 4))

print("F1 drop SVM (LIAR→LIAR vs SVM LIAR→ISOT):",
      round(get_metric("SVM (LIAR→LIAR)") - get_metric("SVM (LIAR→ISOT)"), 4))

print("F1 drop SVM (ISOT→ISOT vs SVM ISOT→LIAR):",
      round(get_metric("SVM (ISOT→ISOT)") - get_metric("SVM (ISOT→LIAR)"), 4))

F1 drop LR (LIAR→LIAR vs LIAR→ISOT): 0.1234
F1 drop LR (ISOT→ISOT vs ISOT→LIAR): 0.9411
F1 drop SVM (LIAR→LIAR vs SVM LIAR→ISOT): 0.083
F1 drop SVM (ISOT→ISOT vs SVM ISOT→LIAR): 0.9514
